# SHOT Phase 2b: Ball Detector Training v3

**Stage 1**: TrackNet 공개 데이터셋(19,835장) 학습
**Stage 2**: 직접 촬영 폰 데이터(809장) Fine-tuning
**Output**: ball_detector.onnx (Save & Run All로 실행)

**Setup:**
1. Add Data: `datasets/faultfoot` (TrackNet) + `phoneBALL` (폰 데이터)
2. Settings → Accelerator → GPU P100
3. Save & Run All

In [ ]:
# 셀 0: PyTorch P100 호환 설치
!pip install torch==2.5.1+cu118 torchvision==0.20.1+cu118 --index-url https://download.pytorch.org/whl/cu118 -q
!pip install onnxruntime -q

In [ ]:
# 셀 1: imports + 경로 확인
import csv, json, os, re, shutil
import numpy as np
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from PIL import Image, ImageFilter

# 경로 확인
TRACKNET_JSON = "/kaggle/input/datasets/faultfoot/balldata/ball_combined.json"
TRACKNET_FRAMES = "/kaggle/input/datasets/faultfoot/balldata/frames"
PHONE_JSON = "/kaggle/input/phoneball/ball_annotations.json"
PHONE_FRAMES = "/kaggle/input/phoneball"
OUTPUT_DIR = "/kaggle/working/models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for name, path in [("TrackNet JSON", TRACKNET_JSON), ("TrackNet frames", TRACKNET_FRAMES),
                    ("Phone JSON", PHONE_JSON), ("Phone frames", PHONE_FRAMES)]:
    ok = os.path.exists(path)
    print(f"  {'[OK]' if ok else '[MISSING]'} {name}: {path}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")

In [ ]:
# 셀 2: 모델 정의

class BallDetector(nn.Module):
    def __init__(self, pretrained=True, dropout=0.15):
        super().__init__()
        weights = MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        backbone = mobilenet_v3_small(weights=weights)
        self.features = backbone.features
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(576, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.Dropout2d(dropout),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.Dropout2d(dropout),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, kernel_size=1),
        )

    def forward(self, x):
        return torch.sigmoid(self.decoder(self.features(x)))


class BallDetectorLoss(nn.Module):
    def __init__(self, alpha=0.97, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred, target):
        bce = F.binary_cross_entropy(pred, target, reduction='none')
        pt = torch.where(target > 0.5, pred, 1 - pred)
        focal_weight = (1 - pt) ** self.gamma
        class_weight = torch.where(target > 0.5, 1.0, 1.0 - self.alpha)
        return (focal_weight * class_weight * bce).mean()


def generate_heatmap(x, y, size=48, sigma=2.5):
    if x < 0 or y < 0:
        return torch.zeros(size, size)
    yy, xx = torch.meshgrid(
        torch.arange(size, dtype=torch.float32),
        torch.arange(size, dtype=torch.float32), indexing='ij')
    return torch.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2 * sigma ** 2))


IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class BallDataset(Dataset):
    INPUT_SIZE = 192
    HEATMAP_SIZE = 48
    SIGMA = 2.5

    def __init__(self, annotations, frames_dir, augment=False):
        self.frames_dir = Path(frames_dir)
        self.augment = augment
        existing = set(os.listdir(str(self.frames_dir))) if self.frames_dir.is_dir() else set()
        self.samples = []
        skipped = 0
        for ann in annotations:
            if existing and ann['image'] not in existing:
                skipped += 1
                continue
            self.samples.append(ann)
        if skipped > 0:
            print(f'  Skipped {skipped} samples (missing frame files)')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ann = self.samples[idx]
        img = Image.open(self.frames_dir / ann['image']).convert('RGB')
        ball_x, ball_y, vis = ann['x'], ann['y'], ann['visibility']

        if self.augment:
            if np.random.random() < 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
                if vis > 0 and ball_x >= 0: ball_x = 1.0 - ball_x
            if np.random.random() < 0.2:
                img = img.filter(ImageFilter.GaussianBlur(radius=np.random.uniform(0.5, 1.5)))

        img = img.resize((self.INPUT_SIZE, self.INPUT_SIZE), Image.BILINEAR)
        img = np.array(img, dtype=np.float32) / 255.0

        if self.augment:
            if np.random.random() < 0.5:
                img = np.clip(img * np.random.uniform(0.7, 1.3), 0, 1)
            if np.random.random() < 0.3:
                img = np.clip(img + np.random.uniform(-0.05, 0.05, size=3).astype(np.float32), 0, 1)
            if np.random.random() < 0.3:
                img = np.clip(img + np.random.normal(0, 0.02, img.shape).astype(np.float32), 0, 1)

        img = (img - IMAGENET_MEAN) / IMAGENET_STD
        img = np.transpose(img, (2, 0, 1))

        if vis > 0 and ball_x >= 0:
            heatmap = generate_heatmap(ball_x * self.HEATMAP_SIZE, ball_y * self.HEATMAP_SIZE, self.HEATMAP_SIZE, self.SIGMA)
        else:
            heatmap = torch.zeros(self.HEATMAP_SIZE, self.HEATMAP_SIZE)

        return torch.from_numpy(img).float(), heatmap.unsqueeze(0).float(), torch.tensor(vis, dtype=torch.long)

print('Model + Dataset defined.')

In [ ]:
# 셀 3: 학습 함수 정의

def train_model(model, train_loader, val_loader, epochs, lr, backbone_lr,
                freeze_epochs, patience, save_prefix, device):
    criterion = BallDetectorLoss()
    optimizer = optim.AdamW([
        {'params': list(model.features.parameters()), 'lr': backbone_lr},
        {'params': list(model.decoder.parameters()), 'lr': lr},
    ], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_error = float('inf')
    no_improve = 0

    print(f"{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>6} | {'VlLoss':>8} | {'VlAcc':>6} | {'VlErr':>6} | {'LR':>8} | Note")
    print('-' * 80)

    for epoch in range(epochs):
        if epoch < freeze_epochs:
            for p in model.features.parameters(): p.requires_grad = False
            note = 'frozen'
        elif epoch == freeze_epochs:
            for p in model.features.parameters(): p.requires_grad = True
            note = 'UNFROZEN'
        else:
            note = ''

        # Train
        model.train()
        train_loss = train_correct = train_total = 0
        for inputs, targets, visibility in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            for i in range(len(visibility)):
                if visibility[i] > 0:
                    train_total += 1
                    if outputs[i].max().item() > 0.5: train_correct += 1

        # Validate
        model.eval()
        val_loss = val_correct = val_total = 0
        position_errors = []
        with torch.no_grad():
            for inputs, targets, visibility in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                val_loss += loss.item()
                for i in range(len(visibility)):
                    if visibility[i] > 0:
                        val_total += 1
                        pred_hm = outputs[i, 0]
                        if pred_hm.max().item() > 0.5:
                            val_correct += 1
                            gt_hm = targets[i, 0]
                            pred_idx = pred_hm.argmax()
                            gt_idx = gt_hm.argmax()
                            h, w = pred_hm.shape
                            err = ((pred_idx % w - gt_idx % w).float() ** 2 +
                                   (pred_idx // w - gt_idx // w).float() ** 2).sqrt().item()
                            position_errors.append(err)

        avg_train_loss = train_loss / max(len(train_loader), 1)
        train_acc = train_correct / max(train_total, 1) * 100
        avg_val_loss = val_loss / max(len(val_loader), 1)
        val_acc = val_correct / max(val_total, 1) * 100
        avg_err = np.mean(position_errors) if position_errors else float('inf')
        scheduler.step()
        lr_now = optimizer.param_groups[1]['lr']

        if avg_err < best_val_error:
            best_val_error = avg_err
            no_improve = 0
            torch.save({'model': model.state_dict(), 'epoch': epoch,
                        'val_loss': avg_val_loss, 'val_error': avg_err},
                       os.path.join(OUTPUT_DIR, f'{save_prefix}_best.pth'))
            note += ' *err'
        else:
            no_improve += 1

        print(f'{epoch:>3} | {avg_train_loss:>8.5f} | {train_acc:>5.1f}% | {avg_val_loss:>8.5f} | {val_acc:>5.1f}% | {avg_err:>5.1f}px | {lr_now:>8.5f} | {note}')

        if no_improve >= patience:
            print(f'\nEarly stopping at epoch {epoch}')
            break

    print(f'Best val_error: {best_val_error:.1f}px (saved as {save_prefix}_best.pth)')
    return best_val_error

print('train_model() defined.')

In [ ]:
# 셀 4: Stage 1 - TrackNet 학습
print('=== STAGE 1: TrackNet Dataset (broadcast) ===')

with open(TRACKNET_JSON) as f:
    tracknet_ann = json.load(f)
print(f'Total: {len(tracknet_ann)}')

# Video-level split
video_frames = defaultdict(list)
for ann in tracknet_ann:
    match = re.match(r'(.+?)_frame_\d+', Path(ann['image']).stem)
    vid_id = match.group(1) if match else Path(ann['image']).stem
    video_frames[vid_id].append(ann)

video_ids = sorted(video_frames.keys())
np.random.seed(42)
np.random.shuffle(video_ids)
split_idx = max(1, int(len(video_ids) * 0.8))

train_ann = [a for vid in video_ids[:split_idx] for a in video_frames[vid]]
val_ann = [a for vid in video_ids[split_idx:] for a in video_frames[vid]]
print(f'Train: {len(train_ann)}, Val: {len(val_ann)}')

train_loader = DataLoader(BallDataset(train_ann, TRACKNET_FRAMES, augment=True),
                          batch_size=32, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(BallDataset(val_ann, TRACKNET_FRAMES, augment=False),
                        batch_size=32, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

model = BallDetector(pretrained=True, dropout=0.15).to(device)
train_model(model, train_loader, val_loader,
            epochs=60, lr=1e-3, backbone_lr=1e-4,
            freeze_epochs=5, patience=20, save_prefix='stage1', device=device)

In [ ]:
# 셀 5: Stage 2 - Phone data fine-tuning
print('\n=== STAGE 2: Phone Data Fine-tuning ===')

# Stage 1 best 모델 로드
ckpt = torch.load(os.path.join(OUTPUT_DIR, 'stage1_best.pth'), map_location=device, weights_only=True)
model.load_state_dict(ckpt['model'])
print(f'Loaded Stage 1 best (epoch {ckpt["epoch"]}, val_error={ckpt["val_error"]:.1f}px)')

with open(PHONE_JSON) as f:
    phone_ann = json.load(f)
print(f'Phone annotations: {len(phone_ann)}')

# Video-level split
video_frames = defaultdict(list)
for ann in phone_ann:
    match = re.match(r'(.+?)_frame_\d+', Path(ann['image']).stem)
    vid_id = match.group(1) if match else Path(ann['image']).stem
    video_frames[vid_id].append(ann)

video_ids = sorted(video_frames.keys())
np.random.seed(42)
np.random.shuffle(video_ids)
split_idx = max(1, int(len(video_ids) * 0.8))

train_ann = [a for vid in video_ids[:split_idx] for a in video_frames[vid]]
val_ann = [a for vid in video_ids[split_idx:] for a in video_frames[vid]]
print(f'Train: {len(train_ann)}, Val: {len(val_ann)}')

train_loader = DataLoader(BallDataset(train_ann, PHONE_FRAMES, augment=True),
                          batch_size=16, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(BallDataset(val_ann, PHONE_FRAMES, augment=False),
                        batch_size=16, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

train_model(model, train_loader, val_loader,
            epochs=30, lr=1e-4, backbone_lr=1e-5,
            freeze_epochs=3, patience=10, save_prefix='stage2', device=device)

In [ ]:
# 셀 6: ONNX 변환
print('\n=== ONNX Export ===')

model_export = BallDetector(pretrained=False, dropout=0.15)
ckpt = torch.load(os.path.join(OUTPUT_DIR, 'stage2_best.pth'), map_location='cpu', weights_only=True)
model_export.load_state_dict(ckpt['model'])
model_export.eval()
print(f'Loaded Stage 2 best (epoch {ckpt["epoch"]}, val_error={ckpt["val_error"]:.1f}px)')

ONNX_PATH = os.path.join(OUTPUT_DIR, 'ball_detector.onnx')
dummy = torch.randn(1, 3, 192, 192)
torch.onnx.export(model_export, dummy, ONNX_PATH, opset_version=18,
                  input_names=['input'], output_names=['heatmap'])
size_mb = os.path.getsize(ONNX_PATH) / 1024 / 1024
print(f'Exported: {ONNX_PATH} ({size_mb:.2f} MB)')

# ONNX 검증
import onnxruntime as ort
session = ort.InferenceSession(ONNX_PATH)
dummy_np = np.random.randn(1, 3, 192, 192).astype(np.float32)
ort_out = session.run(None, {'input': dummy_np})[0]
with torch.no_grad():
    pt_out = model_export(torch.from_numpy(dummy_np)).numpy()
diff = np.abs(pt_out - ort_out).max()
print(f'ONNX shape: {ort_out.shape}, max diff: {diff:.8f}')
print(f'{"[OK]" if diff < 1e-5 else "[WARN]"} PyTorch vs ONNX match')

print('\n=== Output Files ===')
for f in sorted(os.listdir(OUTPUT_DIR)):
    s = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024 / 1024
    print(f'  {f:30s} {s:.2f} MB')